## 1 · Imports & configuration

In [13]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

RAW_PATH = 'ecommerce_iphone_resale_market_intelligence_usa_2026_original.csv'

## 2 · Load raw data

In [14]:
df_raw = pd.read_csv(RAW_PATH)

print(f'Shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

Shape : 2,371 rows × 16 columns


,title,model_family,generation_number,is_pro,storage_options_gb,storage_gb_numeric,condition,price,wasPrice,price_discount_pct,available,sold,seller,itemLocation,us_state,lastUpdated
0,Apple iPhone 14 128gb RED color (Factory Unlocked) 97% battery health,iPhone 14,14.0,False,128GB,128.0,Used,329.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-01-02
1,Lot of 10 Apple iPhone 12 Pro 128GB Unlocked Mixed,iPhone 12 Pro,12.0,True,128GB,128.0,Used,3160.00,NaN,NaN,NaN,NaN,Seller 332,"Winter Park, Florida, United States",FL,2026-03-01
2,Apple iPhone 14 Pro MAX 128GB FULLY Unlocked,iPhone 14 Pro Max,14.0,True,128GB,128.0,Used,726.99,NaN,NaN,3.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
3,Apple iPhone 14 Pro MAX 256gb Space black (Factory Unlocked),iPhone 14 Pro Max,14.0,True,256GB,256.0,Used,669.99,NaN,NaN,NaN,NaN,Seller 019,"Granada Hills, California, United States",CA,2025-12-12
4,Apple iPhone 14 Pro MAX 512gb Deep purple (Factory Unlocked),iPhone 14 Pro Max,14.0,True,512GB,512.0,Used,659.99,NaN,NaN,2.0,NaN,Seller 019,"Granada Hills, California, United States",CA,2026-03-10


In [15]:
# Data types overview
df_raw.dtypes

title                     str
model_family              str
generation_number     float64
is_pro                   bool
storage_options_gb        str
storage_gb_numeric    float64
condition                 str
price                 float64
wasPrice              float64
price_discount_pct    float64
available             float64
sold                  float64
seller                    str
itemLocation              str
us_state                  str
lastUpdated               str
dtype: object

In [16]:
# NULL counts per column
null_counts = df_raw.isnull().sum()
null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': (null_counts / len(df_raw) * 100).round(1)
})
null_summary

,null_count,null_pct
title,0,0.0
model_family,0,0.0
generation_number,0,0.0
is_pro,0,0.0
storage_options_gb,203,8.6
storage_gb_numeric,203,8.6
condition,0,0.0
price,0,0.0
wasPrice,2237,94.3
price_discount_pct,2237,94.3


## 3 · Drop columns not needed for analysis

The columns `seller`, `itemLocation`, `us_state`, and `lastUpdated` do not
contribute to the analytical question (per-unit pricing and market-value
comparisons) and are therefore removed.

In [17]:
COLS_TO_DROP = ['seller', 'itemLocation', 'us_state', 'lastUpdated', 'storage_options_gb']

df = df_raw.drop(columns=COLS_TO_DROP)
print(f'Remaining columns ({len(df.columns)}): {df.columns.tolist()}')

Remaining columns (11): ['title', 'model_family', 'generation_number', 'is_pro', 'storage_gb_numeric', 'condition', 'price', 'wasPrice', 'price_discount_pct', 'available', 'sold']


## 4 · Handle NULL values

| Column | NULL strategy | Rationale |
|---|---|---|
| `storage_gb_numeric` | median imputation | Required for numeric comparisons |
| `wasPrice` | Leave as `NaN` — absence means no discount was advertised | Imputing a "was price" would fabricate discount data |
| `price_discount_pct` | Leave as `NaN` for the same reason | Derived directly from `wasPrice` |
| `available` | Fill `0` — NaN most likely means the listing shows no available stock count | Conservative default; does not distort sold data |
| `sold` | Fill `0` — NaN most likely means the listing shows no sold count | Same rationale |

In [18]:
# ── storage_gb_numeric ───────────────────────────────────────────────────────
# storage_options_gb has been dropped; impute any missing storage_gb_numeric
# values with the column median.
median_storage = df['storage_gb_numeric'].median()
remaining_null = df['storage_gb_numeric'].isnull().sum()
if remaining_null > 0:
    print(f'Imputing {remaining_null} storage_gb_numeric nulls with median = {median_storage:.0f} GB')
    df['storage_gb_numeric'] = df['storage_gb_numeric'].fillna(median_storage)

# ── available & sold ─────────────────────────────────────────────────────────
df['available'] = df['available'].fillna(0)
df['sold'] = df['sold'].fillna(0)

#Comment out if want non-sold stuff.
df = df[df['sold'] > 0]

# ── wasPrice & price_discount_pct ────────────────────────────────────────────
# Intentionally left as NaN — imputing would fabricate discount information.

print('NULL counts after fixing:')
print(df.isnull().sum())

Imputing 203 storage_gb_numeric nulls with median = 128 GB
NULL counts after fixing:
title                   0
model_family            0
generation_number       0
is_pro                  0
storage_gb_numeric      0
condition               0
price                   0
wasPrice              813
price_discount_pct    813
available               0
sold                    0
dtype: int64


## 5 · Normalise "Lot of X" listings to per-unit values

Some titles describe a **multi-unit lot** (e.g., *"Lot of 10 Apple iPhone 12 Pro…"*).
Keeping the bundle price and sold/available counts as-is would skew any
per-unit price analysis.  

**Strategy:**  
1. Detect the lot quantity `X` from the title using a case-insensitive regex
   `Lot [Oo]f (\d+)`.
2. Divide `price`, `wasPrice`, and `price_discount_pct` by `X` so that each
   row represents a single phone.
3. Multiply `sold` and `available` by `X` because the count fields already
   represent the number of *lots* sold/listed, so each lot sold = `X` phones.
4. Recompute `price_discount_pct` from the normalised `wasPrice` and `price`
   where both are available (avoids propagating a division artefact).
5. Add a boolean column `was_lot` to flag these rows for downstream reference.

In [19]:
LOT_PATTERN = re.compile(r'[Ll]ot\s+[Oo]f\s+(\d+)', re.IGNORECASE) #Jank Regex stuff


def extract_lot_quantity(title: str) -> int | None:
    """Return the lot size integer from the listing title, or None."""
    m = LOT_PATTERN.search(str(title))
    return int(m.group(1)) if m else None


# Identify lot rows
df['_lot_qty'] = df['title'].apply(extract_lot_quantity)
lot_mask = df['_lot_qty'].notnull()

print(f'Lot listings detected: {lot_mask.sum()}')
df.loc[lot_mask, ['title', '_lot_qty', 'price', 'sold', 'available']]


Lot listings detected: 2


,title,_lot_qty,price,sold,available
44,Lot of 10 Apple iPhone 12 Unlocked 64gb Mixed Colors,10.0,2110.00,2.0,0.0
151,Lot Of 5 Apple iPhone 14 Plus 128GB (Unlocked),5.0,2049.95,2.0,2.0


In [20]:
# Apply per-unit normalisation
def normalise_lot_row(row):
    """Divide price fields by lot qty; multiply count fields by lot qty."""
    qty = row['_lot_qty']

    # Price fields: bundle price → per-unit price
    row['price'] = row['price'] / qty

    if pd.notnull(row['wasPrice']):
        row['wasPrice'] = row['wasPrice'] / qty

    # Recompute discount pct from normalised prices (avoid division artefact)
    if pd.notnull(row['wasPrice']) and row['wasPrice'] > 0:
        row['price_discount_pct'] = (
            (row['wasPrice'] - row['price']) / row['wasPrice'] * 100
        )
    else:
        row['price_discount_pct'] = np.nan

    # Count fields: lots sold/available → individual phones
    row['sold']      = row['sold']      * qty
    row['available'] = row['available'] * qty

    return row


df.loc[lot_mask] = df.loc[lot_mask].apply(normalise_lot_row, axis=1)

# Flag rows that were originally lot listings
df['was_lot'] = lot_mask

# Drop the temporary helper column
df.drop(columns=['_lot_qty'], inplace=True)

print('Normalised lot rows:')
df.loc[lot_mask, ['title', 'was_lot', 'price', 'sold', 'available']].head(10)

Normalised lot rows:


,title,was_lot,price,sold,available
44,Lot of 10 Apple iPhone 12 Unlocked 64gb Mixed Colors,True,211.00,20.0,0.0
151,Lot Of 5 Apple iPhone 14 Plus 128GB (Unlocked),True,409.99,10.0,10.0


## 6 · Final inspection of the cleaned dataset

In [21]:
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Remaining NULLs:')
print(df.isnull().sum())

Final shape: 912 rows × 12 columns

Remaining NULLs:
title                   0
model_family            0
generation_number       0
is_pro                  0
storage_gb_numeric      0
condition               0
price                   0
wasPrice              813
price_discount_pct    813
available               0
sold                    0
was_lot                 0
dtype: int64


In [22]:
df.describe(include='all')

,title,model_family,generation_number,is_pro,storage_gb_numeric,condition,price,wasPrice,price_discount_pct,available,sold,was_lot
count,912,912,912.000000,912,912.000000,912,912.000000,99.000000,99.000000,912.000000,912.000000,912
unique,897,18,NaN,2,NaN,7,NaN,NaN,NaN,NaN,NaN,2
top,Apple iPhone 17 Pro Max - 1 TB - Cosmic Orange (Unlocked),iPhone 13 Mini,NaN,False,NaN,Used,NaN,NaN,NaN,NaN,NaN,False
freq,3,150,NaN,722,NaN,332,NaN,NaN,NaN,NaN,NaN,910
mean,NaN,NaN,13.856360,NaN,220.872807,NaN,451.539529,846.227778,31.993232,2.902412,161.353070,NaN
std,NaN,NaN,1.551241,NaN,228.798056,NaN,367.618476,547.409014,24.704042,5.530309,1692.210979,NaN
min,NaN,NaN,12.000000,NaN,28.000000,NaN,119.000000,149.950000,2.440000,0.000000,1.000000,NaN
25%,NaN,NaN,13.000000,NaN,128.000000,NaN,234.427500,338.325000,5.000000,0.000000,1.750000,NaN
50%,NaN,NaN,14.000000,NaN,128.000000,NaN,322.990000,799.000000,30.520000,2.000000,4.000000,NaN
75%,NaN,NaN,14.000000,NaN,256.000000,NaN,489.992500,1099.000000,53.960000,4.000000,25.000000,NaN


In [23]:
df.head(10)

,title,model_family,generation_number,is_pro,storage_gb_numeric,condition,price,wasPrice,price_discount_pct,available,sold,was_lot
5,Apple iPhone 14 Pro MAX 1 TB Deep purple (Factory Unlocked),iPhone 14 Pro Max,14.0,True,1024.0,Used,628.99,NaN,NaN,0.0,1.0,False
7,Apple iPhone 14 Pro Max - 128 GB - ALL COLORS (Unlocked)- Very Good Refurb,iPhone 14 Pro Max,14.0,True,128.0,Very Good - Refurbished,469.99,NaN,NaN,5.0,2.0,False
8,open box Apple iPhone 14 Pro MAX 256GB FULLY Unlocked,iPhone 14 Pro Max,14.0,True,256.0,Open Box,729.99,NaN,NaN,0.0,2.0,False
16,Apple iPhone 14 -5G - 512GB Factory Unlocked GSM + CDMA - EXCELLENT,iPhone 14,14.0,False,512.0,Excellent - Refurbished,397.48,NaN,NaN,9.0,30.0,False
24,OPEN BOX Apple iPhone 17 A3258 256GB 512GB BLACK SAGE LAVENDER BLUE (Unlocked),iPhone 17,17.0,False,256.0,Open Box,779.00,NaN,NaN,0.0,48.0,False
27,Apple iPhone 13 128GB Unlocked FACTORY UNLOCKED - EXCELLENT,iPhone 13,13.0,False,128.0,Excellent - Refurbished,249.48,NaN,NaN,0.0,121.0,False
35,open box Apple iPhone 13 128GB Blue Unlocked 1 year ex warranty,iPhone 13,13.0,False,128.0,Open Box,329.99,NaN,NaN,0.0,1.0,False
36,NEW open box Apple iPhone 17 pro max 512GB Unlocked all colors,iPhone 17 Pro Max,17.0,True,512.0,Open Box,1444.99,NaN,NaN,0.0,1.0,False
41,Apple iPhone 12 64GB 128GB 256GB - Unlocked - All Colors - GOOD CONDITION,iPhone 12,12.0,False,64.0,Good - Refurbished,135.99,NaN,NaN,2.0,481.0,False
42,Apple iPhone 14 Pro Max 256GB/512GB/1TB -Deep Purple UNLOCKED,iPhone 14 Pro Max,14.0,True,256.0,Open Box,1119.20,1399.0,20.0,0.0,85.0,False


## 7 · Save cleaned dataset

In [24]:
OUT_PATH = 'ecommerce_iphone_resale_cleaned.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Cleaned dataset saved → {OUT_PATH}')

Cleaned dataset saved → ecommerce_iphone_resale_cleaned.csv
